In [5]:
from ultralytics import YOLO
import torch

In [6]:
from ultralytics import YOLO
import torch

DATA_YAML = "/Users/ilijanasimonovska/Desktop/AMI/yolo26_final_food_dataset/data.yaml"

PROJECT_DIR = "/Users/ilijanasimonovska/Desktop/AMI/runs/detect"

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: False


In [7]:
model_default=YOLO("yolo26s.pt")

In [ ]:
results_26s = model_default.train(
    data=DATA_YAML,
    epochs=150,
    imgsz=768,
    batch=16,
    device="mps",
    project=PROJECT_DIR,
    name="food_yolo26s_default_768",
    patience=20,
    save=True,
    plots=True
)

New https://pypi.org/project/ultralytics/8.4.77 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.75 🚀 Python-3.13.9 torch-2.12.1 MPS (Apple M1 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/ilijanasimonovska/Desktop/AMI/yolo26_final_food_dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1

In [ ]:
PROJECT_DIR

'C:\\Users\\kante\\Documents\\seminarska AMI\\runs\\detect'

In [ ]:
from ultralytics import YOLO

BEST_WEIGHTS = (
    r"C:\Users\kante\Documents\seminarska AMI"
    r"\runs\detect\food_yolo26s_default_768-2\weights\best.pt"
)

model_26s_best = YOLO(BEST_WEIGHTS)

In [ ]:
PROJECT_DIR

'C:\\Users\\kante\\Documents\\seminarska AMI\\runs\\detect'

In [ ]:


test_results_26s = model_26s_best.val(
    data=DATA_YAML,
    split="test",
    imgsz=768,
    batch=16,
    conf=0.25,
    device=0,
    plots=True,
    project=PROJECT_DIR,
    name="food_yolo26s_default_test_768"
)

print("Precision:", test_results_26s.box.mp)
print("Recall:", test_results_26s.box.mr)
print("mAP50:", test_results_26s.box.map50)
print("mAP50-95:", test_results_26s.box.map)

Ultralytics 8.4.46  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 433.8426.0 MB/s, size: 1699.0 KB)
val: Scanning C:\Users\kante\Documents\seminarska AMI\yolo26_final_food_dataset\labels\test.cache... 165 images, 71 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 236/236  0.0s
val: C:\Users\kante\Documents\seminarska AMI\yolo26_final_food_dataset\images\test\1725619767358_image_data.jpg: corrupt JPEG restored and saved
val: C:\Users\kante\Documents\seminarska AMI\yolo26_final_food_dataset\images\test\1725990115268_image_data.jpg: corrupt JPEG restored and saved
val: C:\Users\kante\Documents\seminarska AMI\yolo26_final_food_dataset\images\test\1726086666114_image_data.jpg: corrupt JPEG restored and saved
val: C:\Users\kante\Documents\seminarska AMI\yolo26_final_food_dataset\images\test\1726247200327_image_data.jpg: corrupt JPEG r

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import torch

# ============================================================
# PATHS
# ============================================================
BEST_WEIGHTS = Path(
    r"/Users/ilijanasimonovska/Desktop/AMI/weights/best.pt"
)

TEST_IMAGES_DIR = Path(
    r"/Users/ilijanasimonovska/Desktop/AMI/yolo26_final_food_dataset/images/test"
)

PREDICTIONS_DIR = Path(
    r"/Users/ilijanasimonovska/Desktop/AMI/runs/detect/food_yolo26s_default_predictions_768"
)

CONF_THRESHOLD = 0.10

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

# ============================================================
# VALIDATE PATHS
# ============================================================
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f"Model weights not found:\n{BEST_WEIGHTS}")

if not TEST_IMAGES_DIR.is_dir():
    raise FileNotFoundError(f"Test images folder not found:\n{TEST_IMAGES_DIR}")

print("Model:", BEST_WEIGHTS)
print("Test images:", TEST_IMAGES_DIR)
print("Device:", DEVICE)

# ============================================================
# LOAD MODEL AND RUN PREDICTIONS
# ============================================================
model_26s_best = YOLO(str(BEST_WEIGHTS))

results = model_26s_best.predict(
    source=str(TEST_IMAGES_DIR),
    imgsz=768,
    conf=CONF_THRESHOLD,
    device=DEVICE,

    save=True,          # Save images with bounding boxes
    save_txt=True,      # Save predicted boxes in YOLO format
    save_conf=True,     # Include confidence values in TXT files

    project=str(PREDICTIONS_DIR.parent),
    name=PREDICTIONS_DIR.name,
    exist_ok=True,

    stream=True,        # Memory-efficient processing
    verbose=False
)

# ============================================================
# EXPORT ALL PREDICTIONS TO CSV
# ============================================================
prediction_rows = []
image_summary_rows = []

for result in results:
    filename = Path(result.path).name
    boxes = result.boxes

    number_of_detections = 0 if boxes is None else len(boxes)

    image_summary_rows.append({
        "filename": filename,
        "number_of_detections": number_of_detections
    })

    if boxes is None or len(boxes) == 0:
        continue

    xyxy = boxes.xyxy.cpu().numpy()
    confidences = boxes.conf.cpu().numpy()
    class_ids = boxes.cls.cpu().numpy().astype(int)

    for class_id, confidence, coordinates in zip(
        class_ids, confidences, xyxy
    ):
        x1, y1, x2, y2 = coordinates

        prediction_rows.append({
            "filename": filename,
            "class_id": int(class_id),
            "class_name": result.names[int(class_id)],
            "confidence": float(confidence),
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2)
        })

# ============================================================
# SAVE TABLES
# ============================================================
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

predictions_df = pd.DataFrame(prediction_rows)
summary_df = pd.DataFrame(image_summary_rows)

predictions_csv = PREDICTIONS_DIR / "predictions.csv"
summary_csv = PREDICTIONS_DIR / "prediction_summary_by_image.csv"

predictions_df.to_csv(predictions_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

images_with_detections = (
    summary_df["number_of_detections"] > 0
).sum()

images_without_detections = (
    summary_df["number_of_detections"] == 0
).sum()

print("\nPredictions completed successfully.")
print("=" * 60)
print("Processed test images:        ", len(summary_df))
print("Images with detections:       ", images_with_detections)
print("Images without detections:    ", images_without_detections)
print("Total predicted boxes:        ", len(predictions_df))
print("Confidence threshold:         ", CONF_THRESHOLD)
print("\nAnnotated images saved in:    ", PREDICTIONS_DIR)
print("Prediction table saved in:    ", predictions_csv)
print("Image summary saved in:       ", summary_csv)

Model: C:\Users\kante\Documents\seminarska AMI\runs\detect\food_yolo26s_default_1024-2\weights\best.pt
Test images: C:\Users\kante\Documents\seminarska AMI\yolo26_final_food_dataset\images\test
Device: 0
Results saved to C:\Users\kante\Documents\seminarska AMI\runs\detect\food_yolo26s_default_predictions_1024
173 labels saved to C:\Users\kante\Documents\seminarska AMI\runs\detect\food_yolo26s_default_predictions_1024\labels

Predictions completed successfully.
Processed test images:         236
Images with detections:        168
Images without detections:     68
Total predicted boxes:         340
Confidence threshold:          0.1

Annotated images saved in:     C:\Users\kante\Documents\seminarska AMI\runs\detect\food_yolo26s_default_predictions_1024
Prediction table saved in:     C:\Users\kante\Documents\seminarska AMI\runs\detect\food_yolo26s_default_predictions_1024\predictions.csv
Image summary saved in:        C:\Users\kante\Documents\seminarska AMI\runs\detect\food_yolo26s_defaul